In [0]:
%pip install -r ../requirements-banking.txt

In [0]:
import json
import sys
from pathlib import Path

current = Path.cwd()

project_root = next(
    (
        path
        for path in (current, *current.parents)
        if (path / "config/banking.json").is_file()
        and (path / "src/openbanking").is_dir()
    ),
    None,
)

if project_root is None:
    raise RuntimeError("No se ha encontrado la raíz del proyecto.")

source_path = str(project_root / "src")

if source_path not in sys.path:
    sys.path.insert(0, source_path)

from openbanking.checkpoints import CheckpointStore
from openbanking.enablebanking_client import EnableBankingClient
from openbanking.incremental_banking import run_incremental
from openbanking.storage import Storage

config = json.loads(
    (project_root / "config/banking.json").read_text(
        encoding="utf-8"
    )
)

private_key = dbutils.secrets.get(
    config["secret_scope"],
    config["private_key_secret"],
)

sessions = {
    bank: json.loads(
        dbutils.secrets.get(
            config["secret_scope"],
            bank_config["session_secret"],
        )
    )
    for bank, bank_config in config["banks"].items()
}

storage = Storage(
    account_name=config["storage"]["account_name"],
    account_key=dbutils.secrets.get(
        "azure-storage",
        "graxiano",
    ),
    container_name=config["storage"]["container_name"],
)

checkpoints = CheckpointStore(
    storage=storage,
    prefix=config["storage"]["prefix"],
)

print("Configuración y clientes preparados.")

In [0]:
from datetime import datetime, timezone

date_to = datetime.now(timezone.utc).date()

with EnableBankingClient(
    application_id=config["application_id"],
    private_key=private_key,
    base_url=config["base_url"],
    timeout=config["extraction"]["request_timeout_seconds"],
) as client:

    outcome = run_incremental(
        client=client,
        storage=storage,
        checkpoints=checkpoints,
        sessions=sessions,
        prefix=config["storage"]["prefix"],
        date_to=date_to,
        overlap_days=config["extraction"]["overlap_days"],
    )

print("Estado global:", outcome["status"])

for account in outcome["accounts"]:
    print(
        "\nBanco:", account["bank"],
        "| Cuenta:", account.get("account_index"),
        "| Estado:", account["status"],
    )

    if "date_from" in account:
        print(
            "Rango:",
            account["date_from"],
            "→",
            account["date_to"],
        )

    print(
        "Punto de control guardado:",
        account.get("checkpoint_saved", False),
    )

    if "manifest_path" in account:
        print("Manifiesto:", account["manifest_path"])

    for endpoint in account.get("endpoints", []):
        print(
            endpoint["endpoint"],
            "| Estado:", endpoint["status"],
            "| Registros:", endpoint["records"],
        )

        if "error" in endpoint:
            print("Error:", endpoint["error"])

    if "error" in account:
        print("Error de cuenta:", account["error"])

if outcome["status"] != "success":
    raise RuntimeError(
        "Carga bancaria incompleta. "
        "Revisa los resultados y manifiestos."
    )